# Visualizing Scaled Masked Multi-Head Attention

This document walks through the step-by-step mathematical operations visualized in the Manim animation.

**Parameters:**
- **Sequence Length ($L$):** 3
- **Model Dimension ($d_{model}$):** 4
- **Head Dimension ($d_k$):** 4
- **Scaling Factor:** $1/\sqrt{d_k} = 0.5$

---

### 1. Input Sequence ($X$)
We start with a batch of 3 token embeddings.
**Shape:** $(3, 4)$

| Token | Dim 1 | Dim 2 | Dim 3 | Dim 4 |
| :--- | :---: | :---: | :---: | :---: |
| $t_1$ | 0.1 | -1.2 | 0.5 | 0.2 |
| $t_2$ | 0.8 | 0.3 | -0.1 | 1.1 |
| $t_3$ | -0.5 | 1.5 | 0.9 | 0.4 |

---

### 2. Linear Projections ($Q, K, V$)
The input $X$ is projected using learnable weight matrices ($W_Q, W_K, W_V$) to create Query, Key, and Value matrices.
**Operation:** $Q = XW_Q$, $K = XW_K$, $V = XW_V$

#### Query ($Q$)
| 1.2 | 0.1 | -0.5 |
| :---: | :---: | :---: |
| 0.4 | 1.1 | 0.2 |
| -1.0 | 0.8 | 1.5 |

#### Key ($K$)
| 0.9 | -0.2 | 1.1 |
| :---: | :---: | :---: |
| 1.3 | 0.5 | -0.8 |
| 0.2 | 1.4 | 0.6 |

#### Value ($V$)
| 2.1 | 3.0 | 0.5 |
| :---: | :---: | :---: |
| 1.1 | 0.9 | 4.2 |
| -0.5 | 2.2 | 1.8 |

---

### 3. Raw Scores Calculation
We calculate the similarity between Queries and Keys using a dot product.
**Operation:** $S_{raw} = Q \cdot K^T$

| | $k_1$ | $k_2$ | $k_3$ |
| :--- | :---: | :---: | :---: |
| **$q_1$** | 2.4 | 1.0 | -1.8 |
| **$q_2$** | 1.6 | 3.0 | 0.6 |
| **$q_3$** | -1.0 | 1.8 | 4.0 |

---

### 4. Scaling
To prevent vanishing gradients, scores are scaled by $\frac{1}{\sqrt{d_k}}$.
**Operation:** $S_{scaled} = \frac{S_{raw}}{2.0}$

| | $k_1$ | $k_2$ | $k_3$ |
| :--- | :---: | :---: | :---: |
| **$q_1$** | 1.2 | 0.5 | -0.9 |
| **$q_2$** | 0.8 | 1.5 | 0.3 |
| **$q_3$** | -0.5 | 0.9 | 2.0 |

---

### 5. Causal Masking
To preserve the autoregressive property (tokens cannot attend to the future), we set the upper triangle to negative infinity.

| | $k_1$ | $k_2$ | $k_3$ |
| :--- | :---: | :---: | :---: |
| **$q_1$** | 1.2 | $-\infty$ | $-\infty$ |
| **$q_2$** | 0.8 | 1.5 | $-\infty$ |
| **$q_3$** | -0.5 | 0.9 | 2.0 |

---

### 6. Softmax (Attention Weights)
We apply Softmax row-wise to obtain probability distributions summing to 1.
**Operation:** $A = \text{softmax}(S_{masked})$

| | $k_1$ | $k_2$ | $k_3$ |
| :--- | :---: | :---: | :---: |
| **$q_1$** | **1.0** | 0.0 | 0.0 |
| **$q_2$** | **0.3** | **0.7** | 0.0 |
| **$q_3$** | **0.1** | **0.2** | **0.7** |

---

### 7. Final Output
The attention weights weigh the Value vectors to produce the output.
**Operation:** $Z = A \cdot V$

$$
\begin{bmatrix}
1.0 & 0.0 & 0.0 \\
0.3 & 0.7 & 0.0 \\
0.1 & 0.2 & 0.7
\end{bmatrix}
\times
\begin{bmatrix}
2.1 & 3.0 & 0.5 \\
1.1 & 0.9 & 4.2 \\
-0.5 & 2.2 & 1.8
\end{bmatrix}
=
\begin{bmatrix}
Z_1 \\
Z_2 \\
Z_3
\end{bmatrix}
$$

**Result:**
| Output | Dim 1 | Dim 2 | Dim 3 |
| :--- | :---: | :---: | :---: |
| $Z_1$ | 2.1 | 3.0 | 0.5 |
| $Z_2$ | 1.4 | 1.5 | 3.1 |
| $Z_3$ | 0.1 | 2.0 | 2.1 |

In [1]:
from manim import *
import os

class MaskedMultiHeadAttentionScene(Scene):
    def construct(self):
        # ==========================================
        # 0. CONFIG & SETUP
        # ==========================================
        self.camera.background_color = "#0F172A"  # Dark Slate Blue
        
        # Color Palette
        C_CODE_BG = "#020617"   # Code Panel Background
        C_TEXT_MAIN = "#E2E8F0"
        C_COMMENT = "#64748B"   # Slate 500
        
        # Matrix Colors
        C_X = "#94A3B8"         # Input Gray
        C_W = "#F59E0B"         # Weights Amber
        C_Q = "#38BDF8"         # Query Blue
        C_K = "#34D399"         # Key Emerald
        C_V = "#F472B6"         # Value Pink
        C_S = "#A78BFA"         # Scores Purple
        C_MASK = "#EF4444"      # Red for Mask
        C_OUT = "#FACC15"       # Output Yellow

        # ==========================================
        # 1. CODE PANEL (LEFT SIDE)
        # ==========================================
        # Added Scaling step explicitly
        code_content = """# Dimensions: L=3, d=4
X = torch.randn(3, 4)

# 1. Linear Projections
# Learnable Weights
Q = X @ W_Q 
K = X @ W_K 
V = X @ W_V 

# 2. Raw Scores
S = Q @ K.T

# 3. Scaling (1/sqrt(d))
S = S / 2.0

# 4. Causal Mask
# Upper triangle -> -inf
mask = torch.triu(ones, 1)
S = S.masked_fill(mask, -inf)

# 5. Attention Weights
A = softmax(S, dim=-1)

# 6. Output
Out = A @ V"""
        
        filename = "detailed_attn_scaled.py"
        with open(filename, "w") as f:
            f.write(code_content)

        code_obj = Code(
            filename,
            tab_width=4,
            language="python"
        )
        
        # Style Code Panel
        code_obj.scale(0.55).to_edge(LEFT, buff=0.1)
        if hasattr(code_obj, 'background_mobject'):
            code_obj.background_mobject.set_fill(C_CODE_BG, opacity=0.95)
            code_obj.background_mobject.set_stroke(WHITE, 1)
        
        # Fix comment colors
        code_lines = code_obj.code if hasattr(code_obj, "code") else code_obj[2]
        for i, line_str in enumerate(code_content.split('\n')):
            if line_str.strip().startswith("#"):
                 if i < len(code_lines):
                    code_lines[i].set_color(C_COMMENT)

        self.add(code_obj)

        # Helper: Highlight Code Line
        def highlight_line(line_indices):
            if isinstance(line_indices, int): line_indices = [line_indices]
            lines = code_obj.code if hasattr(code_obj, "code") else code_obj[2]
            max_idx = len(lines) - 1
            safe_indices = [min(i, max_idx) for i in line_indices]
            grp = VGroup(*[lines[i] for i in safe_indices])
            rect = SurroundingRectangle(grp, color=YELLOW, buff=0.02, stroke_width=2)
            return rect

        # Helper: Explanation Text
        exp_txt = Text("Initializing...", font_size=20, color=C_TEXT_MAIN)
        exp_txt.to_edge(DOWN, buff=0.5).set_x(3)
        self.add(exp_txt)

        def update_exp(text):
            new_t = Text(text, font_size=20, color=C_TEXT_MAIN).to_edge(DOWN, buff=0.5).set_x(3)
            return Transform(exp_txt, new_t)

        # ==========================================
        # 2. MATRIX HELPERS
        # ==========================================
        def create_matrix(data, color, label, font_size=18, cell_size=0.5):
            rows, cols = len(data), len(data[0])
            lbl = Text(label, font_size=font_size+2, color=color)
            grid = VGroup()
            for r in range(rows):
                for c in range(cols):
                    square = Square(side_length=cell_size)
                    square.set_stroke(color, 1)
                    square.set_fill(color, opacity=0.1)
                    square.move_to(np.array([c*cell_size, -r*cell_size, 0]))
                    
                    val_str = str(data[r][c])
                    num = Text(val_str, font_size=font_size, color=WHITE)
                    num.move_to(square.get_center())
                    
                    cell = VGroup(square, num)
                    cell.row = r
                    cell.col = c
                    grid.add(cell)
            
            grid.center()
            lbl.next_to(grid, UP, buff=0.1)
            full_obj = VGroup(lbl, grid)
            full_obj.grid = grid
            full_obj.label = lbl
            return full_obj

        # Data Definitions
        data_X = [["0.1", "-1.2", "0.5", "0.2"], 
                  ["0.8", "0.3", "-0.1", "1.1"], 
                  ["-0.5", "1.5", "0.9", "0.4"]] # 3x4
        
        # ==========================================
        # 3. ANIMATION SEQUENCE
        # ==========================================
        
        # STEP 0: Show Input X
        hl = highlight_line([1])
        self.play(Create(hl), update_exp("Start with Input Sequence X"))
        
        matrix_X = create_matrix(data_X, C_X, "Input X")
        matrix_X.scale(0.8).move_to(UP * 2.5 + RIGHT * 0.5)
        self.play(FadeIn(matrix_X))
        self.wait(0.5)

        # STEP 1: Weights & Projections
        hl_new = highlight_line([5,6,7])
        self.play(ReplacementTransform(hl, hl_new), update_exp("1. Multiply X by W_q, W_k, W_v"))

        def get_w_matrix(col, lbl):
            return create_matrix([["."]*3]*4, col, lbl, cell_size=0.4)
        
        w_q = get_w_matrix(C_Q, "W_Q").scale(0.7).next_to(matrix_X, DOWN, buff=0.5).shift(LEFT*2.5)
        w_k = get_w_matrix(C_K, "W_K").scale(0.7).next_to(matrix_X, DOWN, buff=0.5)
        w_v = get_w_matrix(C_V, "W_V").scale(0.7).next_to(matrix_X, DOWN, buff=0.5).shift(RIGHT*2.5)
        
        self.play(FadeIn(w_q), FadeIn(w_k), FadeIn(w_v))
        
        # Arrows
        arr_q = Arrow(matrix_X.get_bottom(), w_q.get_top(), color=C_Q, stroke_width=2)
        arr_k = Arrow(matrix_X.get_bottom(), w_k.get_top(), color=C_K, stroke_width=2)
        arr_v = Arrow(matrix_X.get_bottom(), w_v.get_top(), color=C_V, stroke_width=2)
        
        self.play(GrowArrow(arr_q), GrowArrow(arr_k), GrowArrow(arr_v))

        # Resulting Q, K, V Matrices
        data_Q = [["1.2", "0.1", "-0.5"], ["0.4", "1.1", "0.2"], ["-1.0", "0.8", "1.5"]]
        data_K = [["0.9", "-0.2", "1.1"], ["1.3", "0.5", "-0.8"], ["0.2", "1.4", "0.6"]]
        data_V = [["2.1", "3.0", "0.5"], ["1.1", "0.9", "4.2"], ["-0.5", "2.2", "1.8"]]

        mat_Q = create_matrix(data_Q, C_Q, "Q").scale(0.8).next_to(w_q, DOWN, buff=0.5)
        mat_K = create_matrix(data_K, C_K, "K").scale(0.8).next_to(w_k, DOWN, buff=0.5)
        mat_V = create_matrix(data_V, C_V, "V").scale(0.8).next_to(w_v, DOWN, buff=0.5)

        self.play(
            TransformFromCopy(w_q, mat_Q),
            TransformFromCopy(w_k, mat_K),
            TransformFromCopy(w_v, mat_V)
        )
        self.wait(1)

        # REORGANIZE
        bg_group = VGroup(matrix_X, w_q, w_k, w_v, arr_q, arr_k, arr_v)
        target_y = 2.0
        center_x = 2.0
        
        self.play(
            bg_group.animate.scale(0.5).to_edge(UP, buff=0.1).set_opacity(0.2),
            mat_Q.animate.scale(0.8).move_to(np.array([center_x - 3.5, target_y, 0])),
            mat_K.animate.scale(0.8).move_to(np.array([center_x, target_y, 0])),
            mat_V.animate.scale(0.8).move_to(np.array([center_x + 3.5, target_y, 0])),
        )

        # STEP 2: Raw Scores (Q @ K.T)
        hl2 = highlight_line(10)
        self.play(ReplacementTransform(hl_new, hl2), update_exp("2. Raw Scores = Q @ K.T"))

        # Raw Data (doubled values)
        data_S_raw = [["2.4", "1.0", "-1.8"], 
                      ["1.6", "3.0", "0.6"], 
                      ["-1.0", "1.8", "4.0"]]

        mat_S = create_matrix(data_S_raw, C_S, "Raw Scores").scale(0.9)
        mat_S.move_to(DOWN * 1.5 + LEFT * 0.5)
        
        lines = VGroup(
            Line(mat_Q.get_bottom(), mat_S.get_top(), color=C_S),
            Line(mat_K.get_bottom(), mat_S.get_top(), color=C_S)
        )
        self.play(Create(lines), FadeIn(mat_S))
        self.wait(1)

        # STEP 3: Scaling (Divide by 2)
        hl3 = highlight_line(13)
        self.play(ReplacementTransform(hl2, hl3), update_exp("3. Scale by 1/sqrt(d_k) (Divide by 2)"))

        # Scaled Data (Original intended values)
        data_S_scaled = [["1.2", "0.5", "-0.9"], 
                         ["0.8", "1.5", "0.3"], 
                         ["-0.5", "0.9", "2.0"]]

        # Animate the values changing
        scale_anims = []
        new_label_s = Text("Scaled Scores", font_size=20, color=C_S).move_to(mat_S.label)
        scale_anims.append(Transform(mat_S.label, new_label_s))

        for i, cell in enumerate(mat_S.grid):
            r, c = cell.row, cell.col
            val_str = data_S_scaled[r][c]
            new_txt = Text(val_str, font_size=18, color=WHITE).move_to(cell[1])
            scale_anims.append(Transform(mat_S.grid[i][1], new_txt))
        
        self.play(*scale_anims)
        self.wait(1)

        # STEP 4: Causal Masking
        hl4 = highlight_line([17, 18])
        self.play(
            ReplacementTransform(hl3, hl4), 
            FadeOut(lines),
            update_exp("4. Causal Mask: Set upper triangle to -inf")
        )

        mask_cells = [cell for cell in mat_S.grid if cell.col > cell.row]
        anims = []
        for cell in mask_cells:
            anims.append(cell[0].animate.set_fill(C_MASK, 0.5).set_stroke(C_MASK))
            inf_txt = MathTex(r"-\infty", color=WHITE, font_size=24).move_to(cell[1])
            anims.append(ReplacementTransform(cell[1], inf_txt))
        
        self.play(*anims)
        self.wait(1)

        # STEP 5: Softmax
        hl5 = highlight_line(21)
        self.play(ReplacementTransform(hl4, hl5), update_exp("5. Softmax: Normalize rows"))

        data_A = [["1.0", "0.0", "0.0"], 
                  ["0.3", "0.7", "0.0"], 
                  ["0.1", "0.2", "0.7"]]

        softmax_anims = []
        new_label = Text("Attention (A)", font_size=20, color=C_S).move_to(mat_S.label)
        softmax_anims.append(Transform(mat_S.label, new_label))

        for i, cell in enumerate(mat_S.grid):
            r, c = cell.row, cell.col
            val = data_A[r][c]
            if val == "0.0":
                col, weight = GREY, NORMAL
            else:
                col, weight = GREEN, BOLD
                
            new_txt = Text(val, font_size=18, color=col, weight=weight).move_to(cell[0])
            
            if c > r: # Reset masked cells
                softmax_anims.append(cell[0].animate.set_fill(C_S, 0.1).set_stroke(C_S))
            
            softmax_anims.append(ReplacementTransform(mat_S.grid[i][1], new_txt))
            mat_S.grid[i].remove(mat_S.grid[i][1])
            mat_S.grid[i].add(new_txt)

        self.play(*softmax_anims)
        self.wait(1)

        # STEP 6: Output (A @ V)
        hl6 = highlight_line(24)
        self.play(ReplacementTransform(hl5, hl6), update_exp("6. Output = Attention @ V"))

        # Prevent Overlap: Large buff
        self.play(
            mat_V.animate.next_to(mat_S, RIGHT, buff=1.0)
        )
        
        data_Out = [["Z1"], ["Z2"], ["Z3"]] 
        mat_Out = create_matrix(data_Out, C_OUT, "Out").scale(0.9)
        mat_Out.next_to(mat_V, RIGHT, buff=0.5)
        
        eq = MathTex("=", font_size=30).next_to(mat_V, RIGHT, buff=0.2)
        mat_Out.next_to(eq, RIGHT, buff=0.2)
        
        self.play(Write(eq), FadeIn(mat_Out))
        
        # Highlight Row 1
        row_rect = SurroundingRectangle(VGroup(mat_S.grid[3], mat_S.grid[4], mat_S.grid[5]), color=YELLOW)
        v_rect = SurroundingRectangle(mat_V.grid, color=YELLOW)
        z_rect = SurroundingRectangle(mat_Out.grid[1], color=YELLOW)
        
        self.play(Create(row_rect), Create(v_rect))
        self.play(Transform(row_rect, z_rect), Transform(v_rect, z_rect))
        self.play(FadeOut(row_rect), FadeOut(v_rect), Indicate(mat_Out.grid[1]))

        self.wait(2)
        
        if os.path.exists(filename):
            os.remove(filename)


%manim -qk -v warning MaskedMultiHeadAttentionScene

Manim Community v0.19.0